In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd

In [0]:
print("\n[1/8] Loading table...")

TABLE_NAME = "workspace.default.crime_safety_02_12"

try:
    df = spark.table(TABLE_NAME)
    print(f"✓ Table loaded: {TABLE_NAME}")
except:
    try:
        TABLE_NAME = "default.crime_safety_02_12"
        df = spark.table(TABLE_NAME)
        print(f"Table loaded: {TABLE_NAME}")
    except Exception as e:
        print(f"Error: {e}")
        raise

print("\n--- First 10 Rows ---")
df.show(10, truncate=False)


[1/8] Loading table...
✓ Table loaded: workspace.default.crime_safety_02_12

--- First 10 Rows ---
+----+---------------+------------+------------------+--------------------------------------------------------------------------------+--------+--------------+--------+-------------------+----+-----+-----------+----+--------+----------------------------------+------------------+------------------+----------------------------------------+
|_id |INCIDENT_NUMBER|OFFENSE_CODE|OFFENSE_CODE_GROUP|OFFENSE_DESCRIPTION                                                             |DISTRICT|REPORTING_AREA|SHOOTING|OCCURRED_ON_DATE   |YEAR|MONTH|DAY_OF_WEEK|HOUR|UCR_PART|STREET                            |Lat               |Long              |Location                                |
+----+---------------+------------+------------------+--------------------------------------------------------------------------------+--------+--------------+--------+-------------------+----+-----+-----------+----+----

In [0]:
#Basic Info
print("\n" + "="*70)
print("DATASET OVERVIEW")
print("="*70)
 
row_count = df.count()
col_count = len(df.columns)
 
print(f"Total Rows: {row_count:,}")
print(f"Total Columns: {col_count}")
 
print("\n--- Schema ---")
df.printSchema()
 
print("\n--- Column Names ---")
for i, col_name in enumerate(df.columns, 1):
    print(f"{i:2}. {col_name}")


# Column Analysis
print("\n" + "="*70)
print("DETAILED COLUMN ANALYSIS")
print("="*70)
 
summary_data = []
 
for col_name in df.columns:
    print(f"\n--- {col_name} ---")
 
    col_type = str(df.schema[col_name].dataType)
    print(f"Type: {col_type}")
 
    null_count = df.filter(F.col(col_name).isNull()).count()
    null_pct = (null_count / row_count * 100) if row_count > 0 else 0.0
    print(f"Nulls: {null_count:,} ({null_pct:.2f}%)")
 
    distinct_count = df.select(F.col(col_name)).distinct().count()
    print(f"Distinct: {distinct_count:,}")
 
    try:
        samples = (
            df.select(F.col(col_name))
              .where(F.col(col_name).isNotNull())
              .limit(3)
              .collect()
        )
        sample_values = [str(r[0])[:80] for r in samples]
        print(f"Samples: {sample_values}")
    except Exception:
        print("Samples: (could not retrieve)")
 
    summary_data.append({
        "Column": col_name,
        "Type": col_type.replace("Type", ""),
        "Nulls": int(null_count),
        "Null_Pct": round(float(null_pct), 2),
        "Distinct": int(distinct_count)
    })

# Column summary table
print("\n" + "="*70)
print("COLUMN SUMMARY TABLE")
print("="*70)
 
summary_df = pd.DataFrame(summary_data)
try:
    display(summary_df)  # Databricks display
except Exception:
    print(summary_df.to_string(index=False))


# Numeric Statistics
print("\n" + "="*70)
print("NUMERIC COLUMN STATISTICS")
print("="*70)
 
from pyspark.sql.types import *
 
numeric_cols = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, (DoubleType, IntegerType, LongType, FloatType, DecimalType, ShortType))
]

print(f"Numeric Columns Found: {len(numeric_cols)}")
if numeric_cols:
    print("Columns:", ", ".join(numeric_cols))
    df.select(numeric_cols).describe().show(truncate=False)
else:
    print("No numeric columns found.")


# Missing data
print("\n" + "="*70)
print("MISSING DATA REPORT")
print("="*70)
 
missing_sorted = sorted(summary_data, key=lambda x: x["Null_Pct"], reverse=True)
 
print(f"\n{'Column':<35} {'Missing':<15} {'Percent':<10}")
print("-" * 65)
 
cols_with_missing = 0
for info in missing_sorted:
    if info["Null_Pct"] > 0:
        cols_with_missing += 1
        print(f"{info['Column']:<35} {info['Nulls']:<15,} {info['Null_Pct']:<10.2f}")
 
print(f"\n✓ {cols_with_missing} columns have missing data")

# Duplicate Analysis
print("\n" + "="*70)
print("DUPLICATE ANALYSIS")
print("="*70)
 
distinct_rows = df.distinct().count()
duplicate_rows = row_count - distinct_rows
duplicate_pct = (duplicate_rows / row_count * 100) if row_count > 0 else 0.0
 
print(f"Total Rows:     {row_count:>12,}")
print(f"Distinct Rows:  {distinct_rows:>12,}")
print(f"Duplicates:     {duplicate_rows:>12,} ({duplicate_pct:.2f}%)")

# Categorical frequencies
print("\n" + "="*70)
print("CATEGORICAL COLUMN FREQUENCIES")
print("="*70)
 
categorical_cols = []
for col_name in df.columns:
    if col_name not in numeric_cols:
        try:
            dcount = df.select(F.col(col_name)).distinct().count()
            if 1 < dcount <= 100:
                categorical_cols.append((col_name, dcount))
        except Exception:
            pass
 
print(f"Found {len(categorical_cols)} categorical columns (2–100 unique values).")
for col_name, dcount in categorical_cols[:3]:
    print(f"\n--- {col_name} (distinct: {dcount}) ---")
    df.groupBy(F.col(col_name)).count().orderBy(F.desc("count")).show(10, truncate=False)

# Final Summary
print("\n" + "="*70)
print("PROFILING COMPLETE!")
print("="*70)
 
total_cells = row_count * col_count
null_cells = sum(x["Nulls"] for x in summary_data)
complete_cells = total_cells - null_cells
completeness = (complete_cells / total_cells * 100) if total_cells > 0 else 0.0
 
print(f"\nOVERALL SUMMARY:")
print(f"  • Total Rows:              {row_count:>12,}")
print(f"  • Total Columns:           {col_count:>12,}")
print(f"  • Total Data Points:       {total_cells:>12,}")
print(f"  • Complete Data Points:    {complete_cells:>12,}")
print(f"  • Missing Data Points:     {null_cells:>12,}")
print(f"  • Data Completeness:       {completeness:>11.2f}%")
 
print(f"\nCOLUMN BREAKDOWN:")
print(f"  • Numeric Columns:         {len(numeric_cols):>12}")
print(f"  • Categorical Columns:     {len(categorical_cols):>12}")
print(f"  • Columns with Missing:    {cols_with_missing:>12}")
 
print(f"\nDATA QUALITY:")
print(f"  • Duplicate Rows:          {duplicate_rows:>12,} ({duplicate_pct:.2f}%)")
print(f"  • Unique Rows:             {distinct_rows:>12,}")


DATASET OVERVIEW
Total Rows: 288,945
Total Columns: 18

--- Schema ---
root
 |-- _id: long (nullable = true)
 |-- INCIDENT_NUMBER: string (nullable = true)
 |-- OFFENSE_CODE: long (nullable = true)
 |-- OFFENSE_CODE_GROUP: string (nullable = true)
 |-- OFFENSE_DESCRIPTION: string (nullable = true)
 |-- DISTRICT: string (nullable = true)
 |-- REPORTING_AREA: string (nullable = true)
 |-- SHOOTING: long (nullable = true)
 |-- OCCURRED_ON_DATE: timestamp (nullable = true)
 |-- YEAR: long (nullable = true)
 |-- MONTH: long (nullable = true)
 |-- DAY_OF_WEEK: string (nullable = true)
 |-- HOUR: long (nullable = true)
 |-- UCR_PART: string (nullable = true)
 |-- STREET: string (nullable = true)
 |-- Lat: double (nullable = true)
 |-- Long: double (nullable = true)
 |-- Location: string (nullable = true)


--- Column Names ---
 1. _id
 2. INCIDENT_NUMBER
 3. OFFENSE_CODE
 4. OFFENSE_CODE_GROUP
 5. OFFENSE_DESCRIPTION
 6. DISTRICT
 7. REPORTING_AREA
 8. SHOOTING
 9. OCCURRED_ON_DATE
10. YEAR


Column,Type,Nulls,Null_Pct,Distinct
_id,Long(),42930,14.86,246016
INCIDENT_NUMBER,String(),21636,7.49,251650
OFFENSE_CODE,Long(),42930,14.86,121
OFFENSE_CODE_GROUP,String(),267651,92.63,5635
OFFENSE_DESCRIPTION,String(),42930,14.86,122
DISTRICT,String(),43412,15.02,15
REPORTING_AREA,String(),73529,25.45,883
SHOOTING,Long(),42930,14.86,3
OCCURRED_ON_DATE,Timestamp(),42930,14.86,198732
YEAR,Long(),42930,14.86,5



NUMERIC COLUMN STATISTICS
Numeric Columns Found: 8
Columns: _id, OFFENSE_CODE, SHOOTING, YEAR, MONTH, HOUR, Lat, Long
+-------+-----------------+------------------+--------------------+------------------+------------------+------------------+------------------+-------------------+
|summary|_id              |OFFENSE_CODE      |SHOOTING            |YEAR              |MONTH             |HOUR              |Lat               |Long               |
+-------+-----------------+------------------+--------------------+------------------+------------------+------------------+------------------+-------------------+
|count  |246015           |246015            |246015              |246015            |246015            |246015            |208762            |208762             |
|mean   |123008.0         |2340.3203788386886|0.006885758998435055|2024.0764343637584|6.4604150153445925|12.538146048005203|42.32305490344404 |-71.08380423659266 |
|stddev |71018.55757476352|1175.5228135094856|0.0826944563923